# Logistic Regression - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Logistic Regression?

Logistic regression is a **classification** algorithm (despite its name) used to predict the probability of a binary outcome.

### Mathematical Formulation

#### Sigmoid Function
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

#### Hypothesis
$$h_\theta(x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

#### Cost Function (Binary Cross-Entropy)
$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(h_\theta(x^{(i)})) + (1-y^{(i)}) \log(1-h_\theta(x^{(i)}))]$$

#### Gradient
$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})x_j^{(i)}$$

### Key Properties
- **Output**: Probability between 0 and 1
- **Decision Boundary**: Linear (in feature space)
- **Assumptions**: 
  - Binary outcome (can be extended to multiclass)
  - Independence of observations
  - No multicollinearity
  - Large sample size

### Time Complexity
- Training: O(n_features * n_samples * n_iterations)
- Prediction: O(n_features * n_samples)

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class LogisticRegressionScratch:
    """
    Logistic Regression implementation from scratch.
    
    Parameters:
    -----------
    learning_rate : float, default=0.01
        Learning rate for gradient descent
    n_iterations : int, default=1000
        Number of iterations for gradient descent
    regularization : str or None, default=None
        Type of regularization ('l2', 'l1', or None)
    lambda_reg : float, default=0.01
        Regularization strength
    fit_intercept : bool, default=True
        Whether to calculate the intercept
    verbose : bool, default=False
        Print loss during training
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, 
                 regularization=None, lambda_reg=0.01, 
                 fit_intercept=True, verbose=False):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.regularization = regularization
        self.lambda_reg = lambda_reg
        self.fit_intercept = fit_intercept
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.losses = []
        
    def _sigmoid(self, z):
        """
        Sigmoid activation function.
        Clips input to prevent overflow in exp.
        """
        z = np.clip(z, -500, 500)  # Prevent overflow
        return 1 / (1 + np.exp(-z))
    
    def _initialize_parameters(self, n_features):
        """
        Initialize weights and bias.
        Uses Xavier initialization for better convergence.
        """
        # Xavier initialization
        limit = np.sqrt(2 / n_features)
        self.weights = np.random.uniform(-limit, limit, n_features)
        self.bias = 0 if self.fit_intercept else None
        
    def _compute_loss(self, y_true, y_pred):
        """
        Compute binary cross-entropy loss with optional regularization.
        """
        m = len(y_true)
        
        # Clip predictions to prevent log(0)
        y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
        
        # Binary cross-entropy
        loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        
        # Add regularization
        if self.regularization == 'l2':
            loss += self.lambda_reg / (2 * m) * np.sum(self.weights ** 2)
        elif self.regularization == 'l1':
            loss += self.lambda_reg / m * np.sum(np.abs(self.weights))
            
        return loss
    
    def _gradient_descent(self, X, y, y_pred):
        """
        Perform one step of gradient descent.
        """
        m = len(y)
        
        # Compute gradients
        dw = (1/m) * np.dot(X.T, (y_pred - y))
        
        # Add regularization to weight gradients
        if self.regularization == 'l2':
            dw += (self.lambda_reg / m) * self.weights
        elif self.regularization == 'l1':
            dw += (self.lambda_reg / m) * np.sign(self.weights)
        
        # Update weights
        self.weights -= self.learning_rate * dw
        
        # Update bias if needed
        if self.fit_intercept:
            db = (1/m) * np.sum(y_pred - y)
            self.bias -= self.learning_rate * db
    
    def fit(self, X, y):
        """
        Train the logistic regression model.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values (0 or 1)
        """
        # Convert to numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Initialize parameters
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Forward propagation
            z = np.dot(X, self.weights)
            if self.fit_intercept:
                z += self.bias
            y_pred = self._sigmoid(z)
            
            # Compute loss
            loss = self._compute_loss(y, y_pred)
            self.losses.append(loss)
            
            # Backward propagation
            self._gradient_descent(X, y, y_pred)
            
            # Print progress
            if self.verbose and i % 100 == 0:
                print(f'Iteration {i}, Loss: {loss:.4f}')
        
        return self
    
    def predict_proba(self, X):
        """
        Predict probability for positive class.
        """
        X = np.array(X)
        z = np.dot(X, self.weights)
        if self.fit_intercept:
            z += self.bias
        return self._sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """
        Predict binary class labels.
        """
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """
        Return accuracy score.
        """
        predictions = self.predict(X)
        return np.mean(predictions == y)

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Generate synthetic dataset
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          flip_y=0.1, random_state=42)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Class distribution in training: {np.bincount(y_train)}")

In [ ]:
# Train models with different configurations
models = {
    'No Regularization': LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000),
    'L2 Regularization': LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000, 
                                                   regularization='l2', lambda_reg=0.1),
    'L1 Regularization': LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000, 
                                                   regularization='l1', lambda_reg=0.1)
}

# Train all models
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_scaled, y_train)
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(models.items()):
    axes[idx].plot(model.losses)
    axes[idx].set_title(f'{name} - Loss Curve')
    axes[idx].set_xlabel('Iteration')
    axes[idx].set_ylabel('Loss')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def plot_learning_curves(model_class, X, y, train_sizes=np.linspace(0.1, 1.0, 10)):
    """
    Plot learning curves to diagnose bias/variance.
    """
    train_scores = []
    val_scores = []
    
    for train_size in train_sizes:
        # Split data
        n_samples = int(len(X) * train_size)
        X_subset = X[:n_samples]
        y_subset = y[:n_samples]
        
        # Further split into train/val
        X_tr, X_val, y_tr, y_val = train_test_split(X_subset, y_subset, 
                                                      test_size=0.2, random_state=42)
        
        # Train model
        model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=500)
        model.fit(X_tr, y_tr)
        
        # Evaluate
        train_scores.append(model.score(X_tr, y_tr))
        val_scores.append(model.score(X_val, y_val))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes * len(X), train_scores, 'o-', label='Training Score', linewidth=2)
    plt.plot(train_sizes * len(X), val_scores, 'o-', label='Validation Score', linewidth=2)
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy')
    plt.title('Learning Curves - Diagnosing Bias vs Variance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return train_scores, val_scores

# Plot learning curves
train_scores, val_scores = plot_learning_curves(LogisticRegressionScratch, 
                                                 X_train_scaled, y_train)

In [ ]:
# Comprehensive evaluation
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Comprehensive model evaluation with multiple metrics.
    """
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Confusion Matrix - Train', 'Confusion Matrix - Test',
                       'ROC Curve', 'Precision-Recall Curve')
    )
    
    # Confusion Matrix - Train
    cm_train = confusion_matrix(y_train, y_train_pred)
    fig.add_trace(go.Heatmap(z=cm_train, text=cm_train, texttemplate="%{text}",
                             colorscale='Blues'), row=1, col=1)
    
    # Confusion Matrix - Test
    cm_test = confusion_matrix(y_test, y_test_pred)
    fig.add_trace(go.Heatmap(z=cm_test, text=cm_test, texttemplate="%{text}",
                             colorscale='Blues'), row=1, col=2)
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_test_proba)
    roc_auc = auc(fpr, tpr)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'ROC (AUC = {roc_auc:.3f})'),
                 row=2, col=1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', 
                            line=dict(dash='dash'), showlegend=False),
                 row=2, col=1)
    
    # Precision-Recall Curve
    from sklearn.metrics import precision_recall_curve
    precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
    fig.add_trace(go.Scatter(x=recall, y=precision, name='PR Curve'),
                 row=2, col=2)
    
    # Update layout
    fig.update_layout(height=800, title_text=f"Model Evaluation: {model_name}")
    fig.show()
    
    # Print classification report
    print(f"\nClassification Report for {model_name}:")
    print(classification_report(y_test, y_test_pred))
    
    return cm_test

# Evaluate the best model
best_model = models['L2 Regularization']
cm = evaluate_model(best_model, X_train_scaled, y_train, 
                   X_test_scaled, y_test, 'L2 Regularization')

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """
    Plot decision boundary for 2D data.
    """
    h = 0.02  # Step size in mesh
    
    # Create mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdYlBu, levels=20)
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, 
                         edgecolor='black', s=50)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.colorbar(label='Probability of Class 1')
    
    # Add decision boundary line
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    plt.show()

# Plot decision boundaries for different models
for name, model in models.items():
    plot_decision_boundary(model, X_test_scaled, y_test, 
                          title=f"Decision Boundary - {name}")

In [ ]:
def plot_probability_distribution(model, X, y):
    """
    Plot distribution of predicted probabilities.
    """
    probas = model.predict_proba(X)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Histogram of probabilities by class
    axes[0].hist(probas[y == 0], bins=30, alpha=0.5, label='Class 0', color='blue')
    axes[0].hist(probas[y == 1], bins=30, alpha=0.5, label='Class 1', color='red')
    axes[0].axvline(x=0.5, color='black', linestyle='--', label='Decision Threshold')
    axes[0].set_xlabel('Predicted Probability')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Predicted Probabilities')
    axes[0].legend()
    
    # Box plot
    data_to_plot = [probas[y == 0], probas[y == 1]]
    axes[1].boxplot(data_to_plot, labels=['Class 0', 'Class 1'])
    axes[1].axhline(y=0.5, color='black', linestyle='--', label='Decision Threshold')
    axes[1].set_ylabel('Predicted Probability')
    axes[1].set_title('Box Plot of Predicted Probabilities')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

plot_probability_distribution(best_model, X_test_scaled, y_test)

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Logistic Regression

#### ✅ **Good Use Cases:**
1. **Binary Classification Problems**
   - Customer churn prediction
   - Email spam detection
   - Disease diagnosis (positive/negative)
   - Credit default prediction

2. **Probability Estimation**
   - When you need probability scores, not just class labels
   - Risk assessment applications
   - A/B testing and conversion rate prediction

3. **Interpretability is Important**
   - Feature importance via coefficients
   - Understanding feature relationships
   - Regulatory requirements for model transparency

4. **Baseline Models**
   - Quick first model to establish performance baseline
   - Feature engineering validation

#### ❌ **When NOT to Use:**
1. **Non-linear Relationships**
   - Complex patterns requiring non-linear boundaries
   - Use: Neural Networks, SVM with RBF kernel, Tree-based methods

2. **Multiple Classes (>2)**
   - Can use multinomial logistic regression, but consider:
   - Random Forest, Neural Networks for many classes

3. **Small Sample Size**
   - Requires relatively large samples
   - Use: Naive Bayes, SVM for small datasets

### Data Requirements

#### **Ideal Data Characteristics:**
- **Sample Size**: At least 10-20 samples per feature
- **Feature Types**: Numerical or properly encoded categorical
- **Feature Scaling**: Standardization recommended
- **Class Balance**: Relatively balanced (or use class weights)

#### **Preprocessing Steps:**
1. Handle missing values
2. Encode categorical variables
3. Scale/normalize features
4. Remove multicollinearity
5. Consider polynomial features for non-linearity

### Hyperparameter Tuning Guidelines

| Parameter | Range | Effect |
|-----------|-------|--------|
| Learning Rate | 0.001 - 1.0 | Higher = faster convergence, risk of overshooting |
| Iterations | 100 - 10000 | More = better fit, risk of overfitting |
| Regularization (λ) | 0.001 - 10 | Higher = simpler model, risk of underfitting |
| Threshold | 0.3 - 0.7 | Adjust based on precision/recall requirements |

### Common Pitfalls & Solutions

1. **Perfect Separation**
   - Problem: Coefficients go to infinity
   - Solution: Add regularization

2. **Imbalanced Classes**
   - Problem: Model biased toward majority class
   - Solution: Class weights, resampling, or adjust threshold

3. **Multicollinearity**
   - Problem: Unstable coefficients
   - Solution: Remove correlated features or use L2 regularization

4. **Non-convergence**
   - Problem: Loss doesn't decrease
   - Solution: Adjust learning rate, scale features, check for errors

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare with sklearn implementation
from sklearn.linear_model import LogisticRegression as SklearnLR

# Train both models
our_model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000, 
                                      regularization='l2', lambda_reg=0.1)
our_model.fit(X_train_scaled, y_train)

sklearn_model = SklearnLR(C=10.0, max_iter=1000, random_state=42)
sklearn_model.fit(X_train_scaled, y_train)

# Compare predictions
our_pred = our_model.predict(X_test_scaled)
sklearn_pred = sklearn_model.predict(X_test_scaled)

# Compare probabilities
our_proba = our_model.predict_proba(X_test_scaled)
sklearn_proba = sklearn_model.predict_proba(X_test_scaled)[:, 1]

# Results
print("Performance Comparison:")
print("="*50)
print(f"Our Implementation:")
print(f"  Train Accuracy: {our_model.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {our_model.score(X_test_scaled, y_test):.4f}")
print(f"\nSklearn Implementation:")
print(f"  Train Accuracy: {sklearn_model.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {sklearn_model.score(X_test_scaled, y_test):.4f}")
print(f"\nPrediction Agreement: {np.mean(our_pred == sklearn_pred):.4f}")
print(f"Mean Absolute Probability Difference: {np.mean(np.abs(our_proba - sklearn_proba)):.4f}")

# Plot probability comparison
plt.figure(figsize=(10, 5))
plt.scatter(sklearn_proba, our_proba, alpha=0.5)
plt.plot([0, 1], [0, 1], 'r--', label='Perfect Agreement')
plt.xlabel('Sklearn Probabilities')
plt.ylabel('Our Implementation Probabilities')
plt.title('Probability Predictions Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary & Key Takeaways

### What We Learned:
1. **Mathematical Foundation**: Sigmoid function, cross-entropy loss, gradient descent
2. **Implementation Details**: Proper initialization, numerical stability, regularization
3. **Diagnostics**: Learning curves, confusion matrices, ROC curves
4. **Practical Considerations**: When to use, data requirements, common pitfalls

### Key Insights:
- Logistic regression is simple but powerful for linearly separable data
- Regularization helps prevent overfitting
- Feature scaling is important for convergence
- Threshold tuning can optimize for specific metrics

### Next Steps:
- Try multiclass classification (one-vs-rest or softmax)
- Implement advanced optimizers (Adam, RMSprop)
- Experiment with different initialization strategies
- Add cross-validation for hyperparameter tuning